In [92]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [93]:
data = pd.read_csv("/Users/abhijitdash/Downloads/Bank-data-churn-model.csv")
data.head(10)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,10,15592389,H?,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


In [94]:
## Preprocessing of Data
## Dropping the unnecessary rows
data=data.drop(['RowNumber','CustomerId','Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [95]:
## Encode categorical variables\

gender_label_encoder = LabelEncoder()
data['Gender']= gender_label_encoder.fit_transform(data['Gender'])
data 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [96]:
## Onehot encode 'Geography'
from sklearn.preprocessing import OneHotEncoder
oneHotEncoder_value=OneHotEncoder()
geo_encoder = oneHotEncoder_value.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [97]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [98]:
oneHotEncoder_value.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [99]:
## Convert the Geography as it is a categorical variable so that model must not assume any order or ranking between countries.
encoded_df=pd.DataFrame(geo_encoder.toarray(), columns=oneHotEncoder_value.get_feature_names_out(['Geography']))

In [100]:
encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [101]:
## Combine the onehot ENcoded columns with original columns
data=pd.concat([data.drop('Geography',axis=1),encoded_df], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [102]:
## Save the encoders and scaler so that new customer data is transformed in the same way as training data before making predictions
with open('gender_label_encoder.pkl','wb') as file:
    pickle.dump(gender_label_encoder,file)

with open('oneHotEncoder_value.pkl','wb') as file:
    pickle.dump(oneHotEncoder_value,file)



In [103]:
## Divide the dataset into dependent and independent features- keep exited as dependent and est as independent

X= data.drop('Exited', axis=1)
y=data['Exited']

##Split the data in training and testing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

##Scale these fatures(means bringing all input variables to a similar numerical range so that the model can learn efficiently)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [104]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [105]:
X_test

array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.81311987, -1.09499335,  0.77030065, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.41876609,  0.91324755, -0.94100321, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.24540869,  0.91324755,  0.00972116, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [106]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler,f)

### ANN implementation

In [107]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [108]:
X_train.shape[1]

12

In [109]:
## Build our ANN Model
## Sequential Input → Hidden Layer 1 → Hidden Layer 2 → Output
## Model Architecture 
# Input Layer
#      ↓
# Dense Layer (64 neurons)
#      ↓
# Dense Layer (32 neurons)
#      ↓
# Dense Layer (1 neuron)
#      ↓
# Prediction (Churn / No Churn) 
## Dense---->>. every neuron connects to all neurons from previous layer.
model = Sequential([
   Dense(64, activation='relu',input_shape=(X_train.shape[1],)),  ## !st Hidden layer(HL1) with 64 neurons along with the  with input layer
   Dense(32,activation='relu'),  ###HL2
   Dense(1, activation='sigmoid'), ##OP layer

])

In [110]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 64)                832       
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [111]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

In [112]:
## Compile the model before doing forward and backward propagation
## Before training, model needs 3 things,  
# Optimizer → How weights will be updated, controls how weights change during training.When the model predicts wrong, the optimizer adjusts weights to reduce the error.
# Loss function → How wrong the model is. Since the problem is binary classification 0-> Stays, 1- Leaves
# Metrics → How performance will be measured
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy']) 



In [113]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
##Set up the tensorboard
log_dir = "logs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")  ## store the logs when we train the model
tensorflow_callback=TensorBoard()

In [114]:
## Setup Early Stopping. --EarlyStopping is a TensorFlow callback that stops training automatically when the model stops improving.
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)  ##Watch the validation loss after every epoch
## Wait 10 epochs without improvement before stopping training., After stopping, load the best model weights.

In [119]:
##Train the model
## model.fit()-->Train the neural network using the training data. 
# validation_data=(X_test, y_test) --> After each epoch, evaluate performance on unseen data.
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=10,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/10
250/250 [==============================] - 1s 4ms/step - loss: 0.4342 - accuracy: 0.8087 - val_loss: 0.4167 - val_accuracy: 0.8130
Epoch 2/10
250/250 [==============================] - 1s 4ms/step - loss: 0.4334 - accuracy: 0.8086 - val_loss: 0.4166 - val_accuracy: 0.8145
Epoch 3/10
250/250 [==============================] - 1s 3ms/step - loss: 0.4342 - accuracy: 0.8106 - val_loss: 0.4160 - val_accuracy: 0.8180
Epoch 4/10
250/250 [==============================] - 1s 3ms/step - loss: 0.4340 - accuracy: 0.8094 - val_loss: 0.4183 - val_accuracy: 0.8105
Epoch 5/10
250/250 [==============================] - 1s 3ms/step - loss: 0.4352 - accuracy: 0.8119 - val_loss: 0.4175 - val_accuracy: 0.8135
Epoch 6/10
250/250 [==============================] - 1s 3ms/step - loss: 0.4347 - accuracy: 0.8104 - val_loss: 0.4184 - val_accuracy: 0.8135
Epoch 7/10
250/250 [==============================] - 1s 3ms/step - loss: 0.4369 - accuracy: 0.8104 - val_loss: 0.4241 - val_accuracy: 0.8065
Epoch 

In [120]:
## Save the model file
model.save('model.h5')

In [121]:
## Load Tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [122]:
%tensorboard --logdir logs

Reusing TensorBoard on port 6007 (pid 15226), started 0:10:37 ago. (Use '!kill 15226' to kill it.)